# Report error-analysis figures

Generates the error-analysis CSV files and figures from the outer-fold predictions saved by nested cross-validation.

Inputs:
- `../../csv_analysis/error_analysis_figures/histgradientboosting_oof_predictions.csv`

Outputs:
- `histgradientboosting_per_class_metrics.csv`
- `histgradientboosting_top_confusions.csv`
- Figure 6: normalized confusion matrix
- Figure 7: most frequent misclassification pairs

In [ ]:
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120


def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pipeline.py").exists() and (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "csv_analysis" / "error_analysis_figures"
PREDICTIONS_PATH = OUTPUT_DIR / "histgradientboosting_oof_predictions.csv"
PER_CLASS_PATH = OUTPUT_DIR / "histgradientboosting_per_class_metrics.csv"
TOP_CONFUSIONS_PATH = OUTPUT_DIR / "histgradientboosting_top_confusions.csv"

predictions = pd.read_csv(PREDICTIONS_PATH)
labels = sorted(predictions["true_label"].unique())

print(f"OOF predictions: {len(predictions):,} rows")
print(f"Loaded from: {PREDICTIONS_PATH}")

## Metrics CSVs

Save per-class metrics and the top confused pairs used by the report and SHAP sample selection.

In [ ]:
y_true = predictions["true_label"]
y_pred = predictions["predicted_label"]

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
}

report = classification_report(
    y_true,
    y_pred,
    labels=labels,
    output_dict=True,
    zero_division=0,
)
per_class = pd.DataFrame(report).T.rename_axis("label").reset_index()
per_class.to_csv(PER_CLASS_PATH, index=False)

top_confusions = (
    predictions[predictions["true_label"] != predictions["predicted_label"]]
    .groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
top_confusions.to_csv(TOP_CONFUSIONS_PATH, index=False)

for key, value in metrics.items():
    print(f"{key}: {value * 100:.2f}%")
print(f"Saved to: {PER_CLASS_PATH}")
print(f"Saved to: {TOP_CONFUSIONS_PATH}")

## Figure 6

Normalized confusion matrix over outer-fold predictions.

In [ ]:
cm_norm = confusion_matrix(
    predictions["true_label"],
    predictions["predicted_label"],
    labels=labels,
    normalize="true",
)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm_norm,
    xticklabels=labels,
    yticklabels=labels,
    cmap="Blues",
    square=True,
    cbar=True,
    annot=False,
    linewidths=0.2,
    linecolor="white",
    vmin=0,
    vmax=1,
    ax=ax,
)
ax.set_title("Normalized Confusion Matrix - Outer-Fold Predictions")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
output_path = OUTPUT_DIR / "figure_6_confusion_matrix_normalized.png"
plt.savefig(output_path, dpi=300)
plt.show()
plt.close()
print(f"Saved to: {output_path}")

## Figure 7

Most frequent misclassification pairs.

In [ ]:
plot_df = top_confusions.head(15).copy()
plot_df["pair"] = plot_df["true_label"] + " -> " + plot_df["predicted_label"]

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_df, x="count", y="pair", color="#2979ff", ax=ax)
ax.set_title("Most Frequent Misclassification Pairs")
ax.set_xlabel("Misclassified samples")
ax.set_ylabel("True -> Predicted")
sns.despine(left=True)
plt.tight_layout()
output_path = OUTPUT_DIR / "figure_7_top_confused_pairs.png"
plt.savefig(output_path, dpi=300)
plt.show()
plt.close()
print(f"Saved to: {output_path}")